# Synthetic data contamination pipeline (Colab runner)

Runs stages 00 to 09 on a Colab T4 GPU.

### 1. Check GPU

In [ ]:
import subprocess, sys
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(r.stdout)
    import torch
    assert torch.cuda.is_available(), "CUDA not available"
    print("Using device:", torch.cuda.get_device_name(0))
except Exception as e:
    sys.exit("No GPU detected. Set Runtime -> Change runtime type -> T4 GPU.")


### 2. Clone repo and install requirements

In [ ]:
import os
repo = "synthetic-contamination"
if not os.path.exists(repo):
    !git clone https://github.com/ratik12/synthetic-contamination.git
%cd {repo}
!pip install --upgrade pip
!pip install -r requirements.txt


### 3. Mount Drive and symlink data/ and results/

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

drive_dir = Path("/content/drive/MyDrive/synthetic_contamination_backup")
(drive_dir / "data").mkdir(parents=True, exist_ok=True)
(drive_dir / "results").mkdir(parents=True, exist_ok=True)

for p in ["data", "results"]:
    lp = Path(p)
    if lp.is_symlink() or lp.exists():
        if lp.is_symlink():
            lp.unlink()
    if not lp.exists():
        os.symlink(str(drive_dir / p), p)

print("data ->", os.path.realpath("data"))
print("results ->", os.path.realpath("results"))


### 4. Check environment

In [ ]:
!python scripts/00_check_env.py --config configs/mid.yaml


### 5. Prepare data splits

In [ ]:
!python scripts/01_prepare_data.py --config configs/mid.yaml


### 6. Smoke test generation (16 per generator) and inspect samples

In [ ]:
!python scripts/02_generate.py --config configs/mid.yaml --extra limit=16
!python scripts/show_samples.py --config configs/mid.yaml --raw --n 4


### 7. Run full pipeline (resumes from smoke test)

In [ ]:
!bash run_all.sh configs/mid.yaml


### 8. Make LaTeX tables and figures

In [ ]:
!python scripts/09_make_report_tables.py --config configs/mid.yaml --extra out=report


### 9. View summary metrics

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

f_det = Path("results/mid_today/detection_metrics.csv")
if f_det.exists():
    print("--- Detection metrics ---")
    display(pd.read_csv(f_det))

f_est = Path("results/mid_today/estimator_summary.csv")
if f_est.exists():
    print("--- Estimator summary ---")
    display(pd.read_csv(f_est))
